# Test Pratico: Analisi Vendite 2024
Candidato: Russel Ryan Guevarra

In questo notebook viene eseguita l'analisi dei dati di vendita per l'anno 2024 utilizzando la libreria Pandas.

In [9]:
import pandas as pd

# 1. Caricamento del dataset e pulizia preliminare (Data Cleaning)
df = pd.read_csv('esercizio_it.csv')

# Pulizia stringhe: rimuoviamo spazi bianchi e standardizziamo il testo (Maiuscole/Minuscole)
df['prodotto'] = df['prodotto'].str.strip()
df['nome_agente'] = df['nome_agente'].str.strip().str.title()  # es: "rossi marco" diventa "Rossi Marco"
df['area'] = df['area'].str.strip().str.title()
df['nome_cliente'] = df['nome_cliente'].str.strip()

# Conversione date
df['data_ordine'] = pd.to_datetime(df['data_ordine'], format='mixed', dayfirst=True)

print("="*70)
print(" REPORT ANALISI VENDITE 2024")
print("="*70 + "\n")

# --- RICHIESTA 1: Totale venduto per prodotto ---
print("## 1. TOTALE VENDUTO PER PRODOTTO (Ordinato dal più venduto al meno venduto)\n")
prodotti = df.groupby('prodotto').agg(
    Quantita_Totale=('quantita', 'sum'),
    Fatturato_Totale=('totale', 'sum')
).sort_values(by='Fatturato_Totale', ascending=False)

# Formattazione per la visualizzazione
prodotti_visivi = prodotti.copy()
prodotti_visivi['Fatturato_Totale'] = prodotti_visivi['Fatturato_Totale'].map('€ {:,.2f}'.format)
display(prodotti_visivi)
print("\n" + "-"*70 + "\n")


# --- RICHIESTA 2: Totale venduto per agente e area ---
print("## 2. TOTALE VENDUTO PER AGENTE E PER AREA GEOGRAFICA\n")
agenti_area = df.groupby(['area', 'nome_agente']).agg(
    Fatturato_Totale=('totale', 'sum')
).sort_values(by=['area', 'Fatturato_Totale'], ascending=[True, False])

agenti_visivi = agenti_area.copy()
agenti_visivi['Fatturato_Totale'] = agenti_visivi['Fatturato_Totale'].map('€ {:,.2f}'.format)
display(agenti_visivi)
print("\n" + "-"*70 + "\n")


# --- RICHIESTA 3: Mese migliore e peggiore ---
print("## 3. ANALISI TEMPORALE (Mesi con fatturato più alto e più basso)\n")
df['mese'] = df['data_ordine'].dt.to_period('M')

# Calcolo matematico sui dati numerici puri
fatturato_mensile = df.groupby('mese')['totale'].sum()
mese_max = fatturato_mensile.idxmax()
mese_min = fatturato_mensile.idxmin()

# Creazione del DataFrame di riepilogo con formattazione corretta
report_mesi = pd.DataFrame({
    'Descrizione': ['Mese con fatturato più ALTO 🥇', 'Mese con fatturato più BASSO ⟱'],
    'Mese': [str(mese_max), str(mese_min)],
    'Fatturato': [f"€ {fatturato_mensile[mese_max]:,.2f}", f"€ {fatturato_mensile[mese_min]:,.2f}"]
})
display(report_mesi)
print("\n" + "-"*70 + "\n")


# --- RICHIESTA 4: Clienti con un solo ordine ---
print("## 4. CLIENTI CHE HANNO EFFETTUATO UN SOLO ORDINE NELL'ANNO\n")
ordini_per_cliente = df.groupby(['codice_cliente', 'nome_cliente']).size().reset_index(name='num_ordini')
clienti_unici = ordini_per_cliente[ordini_per_cliente['num_ordini'] == 1]

if clienti_unici.empty:
    print("Nessun cliente ha effettuato un solo ordine nell'anno (tutti i clienti sono ricorrenti).")
else:
    display(clienti_unici[['codice_cliente', 'nome_cliente']])
print("\n" + "="*70)

 REPORT ANALISI VENDITE 2024

## 1. TOTALE VENDUTO PER PRODOTTO (Ordinato dal più venduto al meno venduto)



,Quantita_Totale,Fatturato_Totale
prodotto,,
Antinfiammatorio Iniettabile,263.0,"€ 21,304.20"
Integratore Articolare,250.0,"€ 17,415.62"
Shampoo Dermatologico,257.0,"€ 14,409.32"
Antibiotico Orale,256.0,"€ 14,152.03"
Antiparassitario Spot-On Gatto,210.0,"€ 14,095.39"
Collare Antiparassitario,157.0,"€ 13,132.53"
Antimicotico Topico,210.0,"€ 13,084.65"
Antiparassitario Spot-On Cane,232.0,"€ 12,637.83"
Probiotico Intestinale,159.0,"€ 10,196.85"



----------------------------------------------------------------------

## 2. TOTALE VENDUTO PER AGENTE E PER AREA GEOGRAFICA



Fatturato_Totale
area       nome_agente                      
Centro     Ferrari Giovanni      € 23,760.70
Nord Est   Bianchi Laura         € 27,374.20
           Rossi Marco               € 67.28
Nord Ovest Rossi Marco           € 29,539.28
           Marino Luca           € 29,354.44
           Bianchi Laura            € 646.38
Sud        Conti Sara            € 26,897.59


----------------------------------------------------------------------

## 3. ANALISI TEMPORALE (Mesi con fatturato più alto e più basso)



,Descrizione,Mese,Fatturato
0,Mese con fatturato più ALTO 🥇,2024-10,"€ 16,595.35"
1,Mese con fatturato più BASSO ⟱,2024-06,"€ 8,409.90"



----------------------------------------------------------------------

## 4. CLIENTI CHE HANNO EFFETTUATO UN SOLO ORDINE NELL'ANNO

Nessun cliente ha effettuato un solo ordine nell'anno (tutti i clienti sono ricorrenti).



## Commento sulla Data Quality e Gestione delle Anomalie

Durante l'analisi esplorativa del dataset fittizio `esercizio_it.csv`, sono state riscontrate alcune anomalie strutturali nei dati, risolte in fase di pre-processing tramite codice Python:

1. **Discrepanze di Case-Sensitivity nei Nomi (Agenti e Aree):** Nel dataset originale, alcuni agenti erano registrati con formati di testo diversi (es. `Bianchi Laura`, `BIANCHI LAURA` e `rossi marco`). Senza un intervento, Pandas avrebbe trattato queste righe come commerciali distinti, falsando il calcolo delle performance. I dati sono stati standardizzati applicando la funzione `.str.title()`.
2. **Spazi Bianchi Nascoti (*String Trimming*):**
   È stata rilevata la duplicazione del prodotto `"Antiparassitario Spot-On Cane"`, causata da spazi vuoti inseriti accidentalmente nel testo del file CSV. Il problema è stato risolto applicando la funzione `.str.strip()` a tutte le colonne di tipo testuale prima di procedere con i raggruppamenti (`groupby`).
3. **Integrità del Record-Fatturato:**
   È stato verificato matematicamente che la colonna `totale` corrispondesse al prodotto tra `quantita` e `prezzo_unitario`. Non sono state riscontrate anomalie numeriche o record nulli (*NaN*).